In [1]:
import pandas as pd 
import numpy as np 
from bs4 import BeautifulSoup
import requests
from io import StringIO
import re

ModuleNotFoundError: No module named 'requests'

In [ ]:
url1 = "https://en.wikipedia.org/wiki/List_of_musicals_adapted_into_feature_films"

In [ ]:
headers = {"User-Agent": "MyWikipedeaScraperForACollegeClass_OnHPLaptop15UsingExplorer (jessieaolsen@gmail.com)"}
r = requests.get(url1, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")
r.status_code

In [ ]:
tables = pd.read_html(StringIO(r.text))
adaptations = {}
for table in tables:
    if {'Musical', 'Film'}.issubset(table.columns):
        table['Musical'] = table['Musical'].ffill()

        for _, row in table.dropna(subset=['Film']).iterrows():
            musical = row['Musical']
            if musical == 'Wicked (2003)':
                adaptations.setdefault(musical, []).append('Wicked (2024)')
                adaptations.setdefault(musical, []).append('Wicked: For Good (2025)')

            else:
                film = row['Film']
                adaptations.setdefault(musical, []).append(film)

            

print(adaptations)

In [ ]:
len(adaptations)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

apikey = os.getenv("MY_API_KEY")

rt = {}
imdb = {}
for musical in adaptations:
    for movie in adaptations[musical]:
        title = re.match(r'^(.*?)\s*(\(\d{4}\))$', movie).group(1)
        year = re.match(r'^(.*?)\s*\((\d{4})\)$', movie).group(2)
        print(title, year)
        url = f"http://www.omdbapi.com/?t={title}&y={year}&apikey={apikey}"
        response = requests.get(url).json()
        imdbrating = response.get('imdbRating') if response.get('imdbRating') != 'N/A' else None
        rtrating = None
        for rating in response.get("Ratings", []):
            if rating.get("Source") == "Rotten Tomatoes":
                rtrating = rating.get("Value")
                break
        rt[movie] = rtrating
        imdb[movie] = imdbrating    


In [ ]:
url2 = "https://en.wikipedia.org/wiki/Tony_Award_for_Best_Musical"

In [ ]:
headers = {"User-Agent": "MyWikipedeaScraperForACollegeClass_OnHPLaptop15UsingExplorer (jessieaolsen@gmail.com)"}
r2 = requests.get(url2, headers=headers)
soup2 = BeautifulSoup(r2.text, "html.parser")
r2.status_code

In [ ]:
tables = pd.read_html(StringIO(r2.text))
tonys = []
for table in tables:
    if 'Musical' not in table.columns:
        continue
    table['Musical'] = table['Musical'].ffill()
    for _, row in table.iterrows():
        if pd.notna(row['Musical']):
            match = re.search(r'(.*?)\s*(?:–|-)?\s*(?:Book by|Conceived by|Book & Lyrics)', row['Musical'])
            if match:
                name = match.group(1).strip()
                if name not in tonys:
                    tonys.append(name)


In [ ]:
print(tonys)

In [ ]:
rows = []

for musical, movies in adaptations.items():
    for movie in movies:
        rows.append({"Musical": musical, "Movie": movie})

df = pd.DataFrame(rows)

df["IMDb"] = df["Movie"].map(imdb)
df["RottenTomatoes"] = df["Movie"].map(rt)

df["TonyNominatedMusical"] = df["Musical"].str.extract(r'(.*?)\s\(\d{4}\)')[0].isin(tonys)

df.head(10)

In [ ]:
df.to_csv("musical_adaptations.csv", index=False)

In [ ]:
len(df)